---
format:
  html:
    code-fold: true
jupyter: python3
---

### **Cell 1: Task Definition and Plan**
- Tiny Model: cerebras/Cerebras-GPT-256M. This is a decoder-only transformer small enough for efficient training with minimal memory usage while having sufficient capacity to learn simple associations.
- Toy Task: Opposites
  - Prompt Structure: "The opposite of {input_word} is..."
  - Desired Output: The single word opposite of the inpuut word (e.g., "hot" and "cold" or "high" and "low").

- Reward Logic: Rule-based Gradient Reward Function.
  - Base Reward: 1.0 for finding the correct opposite word.
  - Efficiency Bonus: Up to +1.0 based on how short the response is, aiming for the one word answer.
  - Max Reward: 2.0 (Concise, one word answer)

- Hypothesis: Initially, the model will generate generic or random text. After GRPO fine-tuning, the probability of generating the correct opposite keyword given the input word prompt will maximize, aligning the model with the rule-based function reward.

In [ ]:
# Cell 2: Setup and Model Loading

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
import numpy as np
import random
import copy
from torch.utils.data import Dataset
import re
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_model():
    model_name = "cerebras/Cerebras-GPT-256M"
    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    test_inputs = tokenizer(["Short", "This is a longer sentence"], padding=True, return_tensors="pt")
    print(f"Padding Side: {tokenizer.padding_side}")
    print(f"Input Shape: {test_inputs.input_ids.shape}")
    print(test_inputs.input_ids)

    # Policy Model
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    # Reference Model for KL Divergence
    ref_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    ref_model.eval()
    for param in ref_model.parameters():
        param.requires_grad = False
    return model, ref_model, tokenizer

policy_model, ref_model, tokenizer = load_model()
print(f"Model Loaded: Cerebras-GPT-256M")
print(f"Parameter Count: {policy_model.num_parameters() / 1e6:.1f}M")

In [ ]:
# Cell 3: Dataset Generation

class WordPatternDataset(Dataset):
    def __init__(self, size=500):
        self.pairs = [
            ("up", "down"), ("left", "right"), ("hot", "cold"), ("big", "small"),
            ("good", "bad"), ("happy", "sad"), ("rich", 'poor'), ("fast", "slow"),
            ("high", "low"), ("wet", "dry"), ("hard", "soft"), ("young", "old"),
            ("win", "lose"), ("day", "night"), ("black", "white"), ("in", "out"),
            ("boy", "girl"), ("man", "woman"), ("love", "hate"), ("start", "stop")
        ]
        self.size = size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        input_word, target_word = random.choice(self.pairs)
        prompt = f"The opposite of {input_word} is "

        return {
            "prompt": prompt,
            "target": target_word
        }

# Generate Datasets
train_dataset = WordPatternDataset(size=1000)
eval_dataset = WordPatternDataset(size=50)

print("Word Pattern Dataset Created.")
for i in range(3):
    print(f"Sample {i}: {train_dataset[i]}")

In [ ]:
# Cell 4: Reward Function Implementation

def get_reward(prompts, responses):

    rewards = []

    pairs_map = {
        "up": "down", "left": "right", "hot": "cold", "big": "small",
        "good": "bad", "happy": "sad", "rich": "poor", "fast": "slow",
        "high": "low", "wet": "dry", "hard": "soft", "young": "old",
        "win": "lose", "day": "night", "black": "white", "in": "out",
        "boy": "girl", "man": "woman", "love": "hate", "start": "stop"
    }

    for p, r in zip(prompts, responses):
        try:
            match = re.search(r"opposite of (\w+) is", p)
            if match:
                input_word = match.group(1)
                target_word = pairs_map.get(input_word, "???")

                pred_clean = re.sub(r'[^\w\s]', '', r).strip().lower()
                target_clean = target_word.lower()

                # Check if the target is in the response
                if target_clean in pred_clean:
                    base_score = 1.0

                    # Calculate ratio of Target Length vs. Generated Length
                    gen_len = len(pred_clean) if len(pred_clean) > 0 else 999
                    efficiency_ratio = len(target_clean) / gen_len

                    # Final Reward Score
                    total_score = base_score + efficiency_ratio
                    rewards.append(total_score)

                else:
                    rewards.append(0.0)
            else:
                rewards.append(0.0)

        except Exception:
            rewards.append(0.0)

    return torch.tensor(rewards).float()

# Sanity Check
print("Reward Check")
print(f"Perfect ('cold'): {get_reward(['opposite of hot is'], ['cold']).item():.2f}")
print(f"Repeat   ('cold cold'): {get_reward(['opposite of hot is'], ['cold cold']).item():.2f}")
print(f"Extra Words ('it is cold'): {get_reward(['opposite of hot is'], ['Extra words cold']).item():.2f}")

In [ ]:
# Cell 5: The GRPO Step

def grpo_step(batch_prompts, policy_model, ref_model, tokenizer, device, G=12, beta=0.04):

    group_prompts = [p for p in batch_prompts for _ in range(G)]

    # Tokenize prompts
    inputs = tokenizer(group_prompts, return_tensors="pt", padding=True, padding_side='left').to(device)
    prompt_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = policy_model.generate(
            **inputs,
            max_new_tokens=8,
            min_new_tokens=1,
            do_sample=True,
            temperature=1.2,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.2
        )

    generated_ids = outputs[:, prompt_len:]
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    # Reward
    rewards = get_reward(group_prompts, responses).to(device)

    # Advantage
    rewards_matrix = rewards.view(-1, G)
    group_mean = rewards_matrix.mean(dim=1, keepdim=True)
    group_std = rewards_matrix.std(dim=1, keepdim=True) + 1e-8
    advantages = (rewards_matrix - group_mean) / group_std
    advantages = advantages.view(-1)

    # Attention mask
    attention_mask = (outputs != tokenizer.pad_token_id).long()

    # Policy Model Forward Pass
    policy_out = policy_model(input_ids=outputs, attention_mask=attention_mask)
    policy_logits = policy_out.logits[:, :-1, :]

    # Reference Model Forward Pass
    with torch.no_grad():
        ref_out = ref_model(input_ids=outputs, attention_mask=attention_mask)
        ref_logits = ref_out.logits[:, :-1, :]

    labels = outputs[:, 1:]

    # Log-probabilities for tokens that were generated
    policy_log_probs_all = F.log_softmax(policy_logits, dim=-1)
    ref_log_probs_all = F.log_softmax(ref_logits, dim=-1)

    # Log-probabilities for the actual token that appeared at each position
    policy_log_probs = torch.gather(policy_log_probs_all, -1, labels.unsqueeze(-1)).squeeze(-1)
    ref_log_probs = torch.gather(ref_log_probs_all, -1, labels.unsqueeze(-1)).squeeze(-1)

    # KL Divergence (standard)
    kl_div = policy_log_probs - ref_log_probs

    mask = torch.zeros_like(labels).float()
    mask[:, prompt_len-1:] = 1.0 # Start mask after prompt
    mask = mask * attention_mask[:, 1:] # Apply padding mask

    # GRPO Loss Function
    advantages_expanded = advantages.unsqueeze(1)
    per_token_loss = -1 * (advantages_expanded * policy_log_probs - beta * kl_div)

    masked_loss = per_token_loss * mask
    loss = masked_loss.sum() / mask.sum()

    return loss, rewards.mean().item()

# Output
train_sample = [train_dataset[i]['prompt'] for i in range(2)] # Batch size 2

print(f"Running GRPO Step on: {train_sample}")

loss_val, adv_shape = grpo_step(train_sample, policy_model, ref_model, tokenizer, device)

print(f"Shape of Advantages: {adv_shape}")
print(f"Initial Loss Value: {loss_val.item():.4f}")

In [ ]:
# Cell 6: Training Loop

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=2e-6)
BATCH_SIZE = 4
STEPS_PER_LOG = 10
EPOCHS = 1

dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Starting GRPO Training for {EPOCHS} epoch...")

# Training Loop
step_count = 0
running_loss = 0.0
running_reward = 0.0

policy_model.train()

for epoch in range(EPOCHS):
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in progress_bar:
        prompts = batch['prompt'] # list of strings
        optimizer.zero_grad()

        # GRPO Step
        loss, avg_reward = grpo_step(
            prompts,
            policy_model,
            ref_model,
            tokenizer,
            device,
            G=12,
            beta=0.04
        )

        # Backward Pass & Optimization
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
        optimizer.step()

        # Logging
        step_count += 1
        running_loss += loss.item()
        running_reward += avg_reward

        progress_bar.set_postfix({"Loss": loss.item(), "AvgRew": avg_reward})

        # Periodic detailed logging
        if step_count % STEPS_PER_LOG == 0:
            avg_loss_log = running_loss / STEPS_PER_LOG
            avg_reward_log = running_reward / STEPS_PER_LOG
            tqdm.write(
                f"[Step {step_count}] "
                f"Loss: {avg_loss_log:.4f} | "
                f"Reward: {avg_reward_log:.4f}"
            )

            # Reset running counters
            running_loss = 0.0
            running_reward = 0.0

print("Training Complete")

In [ ]:
# Cell 7: Evaluation and Generation

def evaluate_model(model, dataset, tokenizer, device, batch_size=4, sample_size=None):

    model.eval()
    rewards = []

    if sample_size:
        eval_data = [dataset[i] for i in range(min(sample_size, len(dataset)))]
    else:
        eval_data = dataset

    dataloader = DataLoader(eval_data, batch_size=batch_size)

    print(f"Evaluating on {len(eval_data)} samples...")

    with torch.no_grad():
        for batch in dataloader:
            prompts = batch['prompt']

            # Tokenize
            inputs = tokenizer(prompts, return_tensors="pt", padding=True, padding_side='left').to(device)
            prompt_len = inputs['input_ids'].shape[1]

            # Generate
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

            # Decode
            generated_ids = outputs[:, prompt_len:]
            responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            # Calculate Reward
            batch_rewards = get_reward(prompts, responses)
            rewards.extend(batch_rewards.tolist())

    return sum(rewards) / len(rewards)

print("Quantitative Evaluation")

# Evaluate Baseline
baseline_mean_reward = evaluate_model(ref_model, eval_dataset, tokenizer, device, sample_size=50)

# Evaluate Fine-Tuned
finetuned_mean_reward = evaluate_model(policy_model, eval_dataset, tokenizer, device, sample_size=50)
print(f"Baseline Mean Reward:  {baseline_mean_reward:.4f}")
print(f"Fine-tuned Mean Reward: {finetuned_mean_reward:.4f}")

def Example_generations(model, dataset, tokenizer, device, num_samples=5):
    print(f"\nVisualizing {num_samples} Example Generations")

    for i in range(num_samples):
        sample = dataset[i]
        prompt = sample['prompt']
        target = sample.get('target', 'N/A')


        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
        prompt_len = inputs.input_ids.shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                min_new_tokens=1,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id
            )


        generated_ids = outputs[:, prompt_len:]
        response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        is_success = target.lower() in response.lower()
        status = "✅" if is_success else "❌"

        print(f"\nSample {i+1}:")
        print(f"  Prompt:   '{prompt}'")
        print(f"  Output:   {repr(response)}")
        print(f"  Target:   '{target}'")
        print(f"  Status:   {status}")

Example_generations(policy_model, eval_dataset, tokenizer, device)

### **Cell 8: Analysis**

- Model Evaluation
  - The reward steadily increased per step during the training, as shown by the output of the training cell, and ultimately managed to get close to the maximum allowed reward by the end of the training loop.
  - The model likely went from generating random text sequences (reward ~ 0), to generating text sequences containing the correct word (reward ~ 1). The model then slowly reduced the number of characters in the generated output until it just output the single opposite word (reward ~ 2).

- Dynamics
  - Reward hacking, where the model learns to output just the keyword rather than a full sentence, was observed during the fine-tuning process for this model. This happens because that is the most efficient way to satisfy the get_reward function. The KL penalty and updating the reward function to be more of a gradient-based reward helped mitigate this by anchoring it to the original English distribution while also penalizing the model for unneccesary extra outputs since we only aim to get a single word as an output.
  - Using the GRPO method led to faster convergence/model stability than using PPO would have because it uses the group mean as a baseline, reducing the variance of the gradient estimate without needing a separate Critic network.

- Group Size Impact:
  - The group size used to train the model was 12.
  - It was observed that, with a lower group size, the model has a harder time learning. The reward either stayed below 0.5 or near 0. The lower group size meant the model had fewer choices to pick from during training, making it more difficult to calculate the standard deviation.
  - With a larger group size, the baseline (mean reward) becomes more accurate, leading to cleaner advantage signals, but it consumes significantly more memory. A group size of 12 was a good in-between size.